# Documentation: Declarative Data Pipeline (DLT) Patient Vitals.
### source: Declarative Pipelines/transformations/Patient Vitals.sql

<br>

## 1. Pipeline Architecture
This notebook implements a **Medallion Architecture** using Databricks Delta Live Tables (DLT). It transforms raw physiological JSON data into high-value aggregate insights through three distinct layers: **Bronze**, **Silver**, and **Gold**.

---

## 2. Layer Definitions

### 🥉 Bronze Layer: Raw Ingestion
**Table:** `patient_data.bronze_patient_vitals.patient_vitals`

* **Method:** Uses `read_files` to stream data from the Unity Catalog Volume.
* **Schema Evolution:** Implements a fixed schema for core fields while using a **Rescued Data Column** (`_rescued_data`) to capture unexpected or new fields (like `spo2` or `respiratory_rate`) without breaking the pipeline.
* **Metadata:** Captures `file_name` and `file_modification_time` to provide auditability for every ingested record.

### 🥈 Silver Layer: Cleaning & Validation
**Table:** `patient_data.silver_patient_vitals.patient_vitals_cleaned`

This layer enforces data quality through **Expectations**. If a record violates these clinical bounds, it is dropped to ensure only high-quality data reaches downstream consumers.

| Constraint Name | Logic (Expectation) | Action |
| :--- | :--- | :--- |
| `valid_heart_rate` | $20 < BPM < 300$ | `DROP ROW` |
| `valid_spo2` | $50\% < SpO2 < 100\%$ | `DROP ROW` |
| `valid_resp_rate` | $4 < Resp Rate < 70$ | `DROP ROW` |

* **Transformation:** Casts strings to timestamps and extracts the newly added metrics (`spo2`, `resp_rate`) from the `_rescued_data` JSON column.

### 🥇 Gold Layer: Clinical Aggregates
**Materialized View:** `patient_data.gold_patient_vitals.patient_exercise_summary`

* **Purpose:** Provides a summarized view of patient health indexed by their activity state (`REST`, `EXERCISE`, `CRISIS`).
* **Metrics:** Calculates average Heart Rate, Respiratory Rate, and SpO2. It specifically tracks **Minimum SpO2** to help clinicians identify dangerous "Desaturation" events during physical activity.

---

## 3. Key Declarative Features
* **Streaming Tables:** Automatically manages incremental data processing, ensuring only new files are processed since the last refresh.
* **Managed Lineage:** By using `STREAM {table_name}`, DLT automatically builds the dependency graph between Bronze and Silver.
* **Rescued Data:** The pipeline demonstrates "Schema-on-Read" flexibility by extracting fields from the rescued data column that weren't in the initial Bronze schema definition.

---
